<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/main/Project/Model_Code/Test1_LGBM_visualizations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lightgbm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             classification_report, confusion_matrix, roc_curve,
                             auc, precision_recall_curve)
from scipy import stats
import lightgbm as lgb
import os

# Mount drive
drive.mount('/content/drive')

# ============================================================================
# LOAD DATA
# ============================================================================
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

path = "/content/drive/MyDrive/Sparcs_Datafiles/model_df_clf_v2.feather"
data = pd.read_feather(path)

# Data preprocessing
data['length_of_stay'] = pd.to_numeric(data['length_of_stay'], errors='coerce')
data = data.dropna(subset=['length_of_stay'])
data = data[data['zip_code'] != 'OOS']

# Create log-transformed LOS for regressor
data['los_log'] = np.log1p(data['length_of_stay'])

# Create binary classification target (e.g., LOS > 7 days)
data['long_stay'] = (data['length_of_stay'] > 7).astype(int)

# Define features
categorical_cols = ['health_service_area', 'facility_id', 'zip_code', 'hospital_county',
                    'age_group', 'gender', 'race', 'ethnicity', 'admission_type',
                    'apr_mortality_risk','apr_severity_code', 'apr_drg_code', 'apr_med_surg_desc',
                    'apr_mdc_code', 'ccsr_dx_code', 'ccsr_px_code', 'payment_type']
feature_cols = categorical_cols + ['num_payment_types']

# Encode categorical columns as integers for prediction
from sklearn.preprocessing import LabelEncoder
label_encoders = {}

for col in categorical_cols:
    if col in data.columns:
        le = LabelEncoder()
        # Convert to string first, then encode to int, then convert to float64
        data[col] = le.fit_transform(data[col].astype(str)).astype('float64')
        label_encoders[col] = le

# Ensure num_payment_types is also float64
if 'num_payment_types' in data.columns:
    data['num_payment_types'] = data['num_payment_types'].astype('float64')

X = data[feature_cols].copy()
# Force all columns to be float64 and remove any categorical dtype
for col in X.columns:
    X[col] = X[col].astype('float64')
y_reg = data['los_log']
y_clf = data['long_stay']

# Train-test split (same random state as original)
from sklearn.model_selection import train_test_split
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

print(f"✓ Data loaded: {len(data):,} samples")
print(f"✓ Train size: {len(X_train):,} | Test size: {len(X_test):,}")

# ============================================================================
# LOAD MODELS
# ============================================================================
print("\n" + "="*60)
print("LOADING MODELS")
print("="*60)

# Load LGBM Classifier
lgbm_clf = lgb.Booster(model_file='/content/drive/MyDrive/Sparcs_Datafiles/Hanna - Saved Models/first iterations (regressor and classifier)/lightgbm_classifier_model.txt')
print("✓ LGBM Classifier loaded")

# Load LGBM Regressor
lgbm_reg = lgb.Booster(model_file='/content/drive/MyDrive/Sparcs_Datafiles/Hanna - Saved Models/first iterations (regressor and classifier)/lightgbm_regressor_model.txt')
print("✓ LGBM Regressor loaded")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================
print("\n" + "="*60)
print("GENERATING PREDICTIONS")
print("="*60)

# Convert to pure numpy arrays to avoid pandas categorical issues
X_test_numpy = X_test.values.astype('float64')

# Regressor predictions (using numpy array)
y_reg_pred_log = lgbm_reg.predict(X_test_numpy)
y_reg_pred = np.expm1(y_reg_pred_log)
y_reg_true = np.expm1(y_reg_test.values)

# Classifier predictions (using numpy array)
y_clf_pred_proba = lgbm_clf.predict(X_test_numpy)
y_clf_pred = (y_clf_pred_proba > 0.5).astype(int)

# Calculate metrics
reg_mae = mean_absolute_error(y_reg_true, y_reg_pred)
reg_rmse = np.sqrt(mean_squared_error(y_reg_true, y_reg_pred))
reg_r2 = r2_score(y_reg_true, y_reg_pred)

print(f"\nRegressor Metrics:")
print(f"  MAE:  {reg_mae:.3f} days")
print(f"  RMSE: {reg_rmse:.3f} days")
print(f"  R²:   {reg_r2:.3f}")

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
clf_acc = accuracy_score(y_clf_test, y_clf_pred)
clf_prec = precision_score(y_clf_test, y_clf_pred)
clf_rec = recall_score(y_clf_test, y_clf_pred)
clf_f1 = f1_score(y_clf_test, y_clf_pred)

print(f"\nClassifier Metrics:")
print(f"  Accuracy:  {clf_acc:.3f}")
print(f"  Precision: {clf_prec:.3f}")
print(f"  Recall:    {clf_rec:.3f}")
print(f"  F1-Score:  {clf_f1:.3f}")

# ============================================================================
# VISUALIZATION SETUP
# ============================================================================
output_dir = "/content/downloads/"
os.makedirs(output_dir, exist_ok=True)

print("\n" + "="*60)
print("GENERATING VISUALIZATIONS")
print("="*60)

# ============================================================================
# REGRESSOR VISUALIZATIONS
# ============================================================================

# 1. PREDICTED VS ACTUAL (REGRESSOR)
print("\n[REGRESSOR 1/6] Predicted vs Actual...")

fig, ax = plt.subplots(figsize=(10, 10))

# Sample for performance if needed
if len(y_reg_test) > 50000:
    idx = np.random.choice(len(y_reg_test), 50000, replace=False)
    y_true_plot = y_reg_true[idx]
    y_pred_plot = y_reg_pred[idx]
else:
    y_true_plot = y_reg_true
    y_pred_plot = y_reg_pred

# Create density coloring
xy = np.vstack([y_true_plot, y_pred_plot])
z = stats.gaussian_kde(xy)(xy)
idx_sort = z.argsort()

scatter = ax.scatter(
    y_true_plot[idx_sort],
    y_pred_plot[idx_sort],
    c=z[idx_sort], cmap='viridis', s=15, alpha=0.5
)

# Perfect prediction line
max_val = max(y_reg_true.max(), y_reg_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel("Actual LOS (days)", fontsize=12, fontweight='bold')
ax.set_ylabel("Predicted LOS (days)", fontsize=12, fontweight='bold')
ax.set_title(f"LGBM Regressor: Predicted vs Actual LOS\nMAE: {reg_mae:.3f} days | R²: {reg_r2:.4f}",
             fontsize=15, fontweight='bold')
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, label="Point Density")
plt.tight_layout()
plt.savefig(f"{output_dir}reg_01_predicted_vs_actual.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved reg_01_predicted_vs_actual.png")

# 2. RESIDUAL ANALYSIS (REGRESSOR)
print("[REGRESSOR 2/6] Residual Analysis...")

residuals = y_reg_true - y_reg_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("LGBM Regressor: Residual Analysis", fontsize=16, fontweight='bold')

# Sample for performance
if len(residuals) > 50000:
    idx = np.random.choice(len(residuals), 50000, replace=False)
    residuals_plot = residuals[idx]
    y_pred_plot = y_reg_pred[idx]
else:
    residuals_plot = residuals
    y_pred_plot = y_reg_pred

# 2A: Residuals vs Predicted
axes[0,0].scatter(y_pred_plot, residuals_plot, alpha=0.3, s=8, color='steelblue')
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0,0].set_xlabel("Predicted LOS (days)", fontweight='bold')
axes[0,0].set_ylabel("Residuals (days)", fontweight='bold')
axes[0,0].set_title("Residuals vs Predicted", fontweight='bold')
axes[0,0].grid(True, alpha=0.3)

# 2B: Histogram
axes[0,1].hist(residuals, bins=100, alpha=0.7, edgecolor='black', color='steelblue')
axes[0,1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_xlabel("Residuals (days)", fontweight='bold')
axes[0,1].set_ylabel("Frequency", fontweight='bold')
axes[0,1].set_title("Residual Distribution", fontweight='bold')
axes[0,1].grid(True, alpha=0.3, axis='y')

mean_res = np.mean(residuals)
std_res = np.std(residuals)
axes[0,1].text(0.02, 0.98, f'Mean: {mean_res:.3f}\nStd: {std_res:.3f}',
               transform=axes[0,1].transAxes, fontsize=10,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2C: Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[1,0])
axes[1,0].set_title("Normal Q-Q Plot", fontweight='bold')
axes[1,0].grid(True, alpha=0.3)

# 2D: Absolute Residuals vs Predicted
axes[1,1].scatter(y_pred_plot, np.abs(residuals_plot), alpha=0.3, s=8, color='steelblue')
axes[1,1].set_xlabel("Predicted LOS (days)", fontweight='bold')
axes[1,1].set_ylabel("Absolute Residuals (days)", fontweight='bold')
axes[1,1].set_title("Absolute Residuals vs Predicted", fontweight='bold')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{output_dir}reg_02_residual_analysis.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved reg_02_residual_analysis.png")

# 3. ERROR BY LOS BINS (REGRESSOR)
print("[REGRESSOR 3/6] Error by LOS bins...")

errors = y_reg_pred - y_reg_true
bins = [0, 2, 4, 7, 14, 30, y_reg_true.max()+1]
labels = ['0-2', '3-4', '5-7', '8-14', '15-30', '30+']
y_bin = pd.cut(y_reg_true, bins=bins, labels=labels)

df_error = pd.DataFrame({
    "Bin": y_bin,
    "Error": errors,
    "Abs_Error": np.abs(errors)
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("LGBM Regressor: Error Distribution by Actual LOS Range",
             fontsize=16, fontweight='bold')

sns.boxplot(x="Bin", y="Error", data=df_error, ax=axes[0], palette='Set2')
axes[0].axhline(0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel("Actual LOS Range (days)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Prediction Error (days)", fontsize=12, fontweight='bold')
axes[0].set_title("Error Distribution (Predicted - Actual)", fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

sns.boxplot(x="Bin", y="Abs_Error", data=df_error, ax=axes[1], palette='Set3')
axes[1].set_xlabel("Actual LOS Range (days)", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Absolute Prediction Error (days)", fontsize=12, fontweight='bold')
axes[1].set_title("Absolute Error Distribution", fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{output_dir}reg_03_error_by_los_bins.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved reg_03_error_by_los_bins.png")

# 4. FEATURE IMPORTANCE (REGRESSOR)
print("[REGRESSOR 4/6] Feature Importance...")

importances = lgbm_reg.feature_importance()
df_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
})
df_imp = df_imp.sort_values("Importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(y="Feature", x="Importance", data=df_imp, ax=ax,
            palette="Blues_r", edgecolor='black', linewidth=1.5)
ax.set_xlabel("Importance", fontsize=12, fontweight='bold')
ax.set_ylabel("Feature", fontsize=12, fontweight='bold')
ax.set_title("LGBM Regressor: Top 20 Feature Importances", fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f"{output_dir}reg_04_feature_importance.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved reg_04_feature_importance.png")

# 5. PREDICTION DISTRIBUTIONS (REGRESSOR)
print("[REGRESSOR 5/6] Prediction Distributions...")

def categorize_los(days):
    if days <= 2:
        return 0
    elif days <= 7:
        return 1
    elif days <= 14:
        return 2
    else:
        return 3

# Apply categorization to numpy array
y_test_category = np.array([categorize_los(d) for d in y_reg_true])
category_names = ['Short (1-2d)', 'Moderate (3-7d)', 'Long (8-14d)', 'Extreme (15+d)']

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("LGBM Regressor: Prediction Distributions by True Category",
             fontsize=16, fontweight='bold')
axs = axs.ravel()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

for cat in range(4):
    mask = (y_test_category == cat)

    axs[cat].hist(y_reg_true[mask], bins=50, alpha=0.5, label="Actual",
                 color=colors[cat], edgecolor='black')
    axs[cat].hist(y_reg_pred[mask], bins=50, alpha=0.5, label="Predicted",
                 color="gray", edgecolor='black')
    axs[cat].set_xlabel("LOS (days)", fontweight='bold')
    axs[cat].set_ylabel("Frequency", fontweight='bold')
    axs[cat].set_title(f"{category_names[cat]} (n={mask.sum():,})", fontweight='bold')
    axs[cat].legend(loc='upper right')
    axs[cat].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{output_dir}reg_05_prediction_distributions.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved reg_05_prediction_distributions.png")

# 6. MODEL SUMMARY TABLE (REGRESSOR)
print("[REGRESSOR 6/6] Model Summary Table...")

summary_reg = pd.DataFrame({
    "Metric": [
        "Test Samples",
        "MAE (days)",
        "RMSE (days)",
        "R²",
        "Mean Actual LOS",
        "Mean Predicted LOS",
        "Median Actual",
        "Median Predicted",
        "Std Actual",
        "Std Predicted"
    ],
    "Value": [
        f"{len(y_reg_test):,}",
        f"{reg_mae:.3f}",
        f"{reg_rmse:.3f}",
        f"{reg_r2:.4f}",
        f"{y_reg_true.mean():.2f}",
        f"{y_reg_pred.mean():.2f}",
        f"{np.median(y_reg_true):.1f}",
        f"{np.median(y_reg_pred):.1f}",
        f"{y_reg_true.std():.2f}",
        f"{y_reg_pred.std():.2f}"
    ]
})

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis("tight")
ax.axis("off")

table = ax.table(
    cellText=summary_reg.values,
    colLabels=summary_reg.columns,
    loc="center",
    cellLoc="left",
    colWidths=[0.6, 0.4]
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for i in range(len(summary_reg.columns)):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(weight='bold', color='white')

for i in range(1, len(summary_reg) + 1):
    if i % 2 == 0:
        for j in range(len(summary_reg.columns)):
            table[(i, j)].set_facecolor('#E7E6E6')

plt.title("LGBM Regressor: Performance Summary", fontsize=14, fontweight='bold', pad=20)
plt.savefig(f"{output_dir}reg_06_model_summary.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved reg_06_model_summary.png")

# ============================================================================
# CLASSIFIER VISUALIZATIONS
# ============================================================================

# 1. CONFUSION MATRIX (CLASSIFIER)
print("\n[CLASSIFIER 1/6] Confusion Matrix...")

cm = confusion_matrix(y_clf_test, y_clf_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax,
            xticklabels=['Short Stay', 'Long Stay'],
            yticklabels=['Short Stay', 'Long Stay'])
ax.set_xlabel("Predicted", fontsize=12, fontweight='bold')
ax.set_ylabel("Actual", fontsize=12, fontweight='bold')
ax.set_title(f"LGBM Classifier: Confusion Matrix\nAccuracy: {clf_acc:.3f}",
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{output_dir}clf_01_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved clf_01_confusion_matrix.png")

# 2. ROC CURVE (CLASSIFIER)
print("[CLASSIFIER 2/6] ROC Curve...")

fpr, tpr, _ = roc_curve(y_clf_test, y_clf_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('LGBM Classifier: ROC Curve', fontsize=14, fontweight='bold')
ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{output_dir}clf_02_roc_curve.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved clf_02_roc_curve.png")

# 3. PRECISION-RECALL CURVE (CLASSIFIER)
print("[CLASSIFIER 3/6] Precision-Recall Curve...")

precision, recall, _ = precision_recall_curve(y_clf_test, y_clf_pred_proba)
pr_auc = auc(recall, precision)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(recall, precision, color='blue', lw=2,
        label=f'PR curve (AUC = {pr_auc:.3f})')
ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('LGBM Classifier: Precision-Recall Curve', fontsize=14, fontweight='bold')
ax.legend(loc="lower left", fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig(f"{output_dir}clf_03_precision_recall.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved clf_03_precision_recall.png")

# 4. FEATURE IMPORTANCE (CLASSIFIER)
print("[CLASSIFIER 4/6] Feature Importance...")

importances_clf = lgbm_clf.feature_importance()
df_imp_clf = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances_clf
})
df_imp_clf = df_imp_clf.sort_values("Importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(y="Feature", x="Importance", data=df_imp_clf, ax=ax,
            palette="Reds_r", edgecolor='black', linewidth=1.5)
ax.set_xlabel("Importance", fontsize=12, fontweight='bold')
ax.set_ylabel("Feature", fontsize=12, fontweight='bold')
ax.set_title("LGBM Classifier: Top 20 Feature Importances", fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f"{output_dir}clf_04_feature_importance.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved clf_04_feature_importance.png")

# 5. PREDICTION PROBABILITY DISTRIBUTION (CLASSIFIER)
print("[CLASSIFIER 5/6] Prediction Probability Distribution...")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("LGBM Classifier: Prediction Probability Distributions",
             fontsize=16, fontweight='bold')

# Distribution by true class
for true_class in [0, 1]:
    mask = (y_clf_test == true_class)
    label = 'Short Stay' if true_class == 0 else 'Long Stay'
    axes[0].hist(y_clf_pred_proba[mask], bins=50, alpha=0.5, label=f'True: {label}',
                edgecolor='black')

axes[0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold')
axes[0].set_xlabel("Predicted Probability", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Frequency", fontsize=12, fontweight='bold')
axes[0].set_title("Distribution by True Class", fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Overall distribution
axes[1].hist(y_clf_pred_proba, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold')
axes[1].set_xlabel("Predicted Probability", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Frequency", fontsize=12, fontweight='bold')
axes[1].set_title("Overall Distribution", fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{output_dir}clf_05_probability_distribution.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved clf_05_probability_distribution.png")

# 6. MODEL SUMMARY TABLE (CLASSIFIER)
print("[CLASSIFIER 6/6] Model Summary Table...")

summary_clf = pd.DataFrame({
    "Metric": [
        "Test Samples",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC AUC",
        "PR AUC",
        "True Negatives",
        "False Positives",
        "False Negatives",
        "True Positives"
    ],
    "Value": [
        f"{len(y_clf_test):,}",
        f"{clf_acc:.3f}",
        f"{clf_prec:.3f}",
        f"{clf_rec:.3f}",
        f"{clf_f1:.3f}",
        f"{roc_auc:.3f}",
        f"{pr_auc:.3f}",
        f"{cm[0,0]:,}",
        f"{cm[0,1]:,}",
        f"{cm[1,0]:,}",
        f"{cm[1,1]:,}"
    ]
})

fig, ax = plt.subplots(figsize=(8, 6))
ax.axis("tight")
ax.axis("off")

table = ax.table(
    cellText=summary_clf.values,
    colLabels=summary_clf.columns,
    loc="center",
    cellLoc="left",
    colWidths=[0.6, 0.4]
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for i in range(len(summary_clf.columns)):
    table[(0, i)].set_facecolor('#C44444')
    table[(0, i)].set_text_props(weight='bold', color='white')

for i in range(1, len(summary_clf) + 1):
    if i % 2 == 0:
        for j in range(len(summary_clf.columns)):
            table[(i, j)].set_facecolor('#E7E6E6')

plt.title("LGBM Classifier: Performance Summary", fontsize=14, fontweight='bold', pad=20)
plt.savefig(f"{output_dir}clf_06_model_summary.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✓ Saved clf_06_model_summary.png")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*60)
print("VISUALIZATION COMPLETE")
print("="*60)
print(f"\n✓ All visualizations saved to: {output_dir}")
print(f"\nRegressor Files:")
print("  • reg_01_predicted_vs_actual.png")
print("  • reg_02_residual_analysis.png")
print("  • reg_03_error_by_los_bins.png")
print("  • reg_04_feature_importance.png")
print("  • reg_05_prediction_distributions.png")
print("  • reg_06_model_summary.png")
print(f"\nClassifier Files:")
print("  • clf_01_confusion_matrix.png")
print("  • clf_02_roc_curve.png")
print("  • clf_03_precision_recall.png")
print("  • clf_04_feature_importance.png")
print("  • clf_05_probability_distribution.png")
print("  • clf_06_model_summary.png")
print("\n" + "="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

LOADING DATA
✓ Data loaded: 4,110,661 samples
✓ Train size: 3,288,528 | Test size: 822,133

LOADING MODELS
✓ LGBM Classifier loaded
✓ LGBM Regressor loaded

GENERATING PREDICTIONS

Regressor Metrics:
  MAE:  2.775 days
  RMSE: 6.731 days
  R²:   0.410

Classifier Metrics:
  Accuracy:  0.231
  Precision: 0.204
  Recall:    0.976
  F1-Score:  0.337

GENERATING VISUALIZATIONS

[REGRESSOR 1/6] Predicted vs Actual...
   ✓ Saved reg_01_predicted_vs_actual.png
[REGRESSOR 2/6] Residual Analysis...
   ✓ Saved reg_02_residual_analysis.png
[REGRESSOR 3/6] Error by LOS bins...


/tmp/ipython-input-1656209645.py:265: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="Bin", y="Error", data=df_error, ax=axes[0], palette='Set2')
/tmp/ipython-input-1656209645.py:272: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="Bin", y="Abs_Error", data=df_error, ax=axes[1], palette='Set3')


   ✓ Saved reg_03_error_by_los_bins.png
[REGRESSOR 4/6] Feature Importance...


/tmp/ipython-input-1656209645.py:294: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(y="Feature", x="Importance", data=df_imp, ax=ax,


   ✓ Saved reg_04_feature_importance.png
[REGRESSOR 5/6] Prediction Distributions...
   ✓ Saved reg_05_prediction_distributions.png
[REGRESSOR 6/6] Model Summary Table...
   ✓ Saved reg_06_model_summary.png

[CLASSIFIER 1/6] Confusion Matrix...
   ✓ Saved clf_01_confusion_matrix.png
[CLASSIFIER 2/6] ROC Curve...
   ✓ Saved clf_02_roc_curve.png
[CLASSIFIER 3/6] Precision-Recall Curve...
   ✓ Saved clf_03_precision_recall.png
[CLASSIFIER 4/6] Feature Importance...


/tmp/ipython-input-1656209645.py:486: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(y="Feature", x="Importance", data=df_imp_clf, ax=ax,


   ✓ Saved clf_04_feature_importance.png
[CLASSIFIER 5/6] Prediction Probability Distribution...
   ✓ Saved clf_05_probability_distribution.png
[CLASSIFIER 6/6] Model Summary Table...
   ✓ Saved clf_06_model_summary.png

VISUALIZATION COMPLETE

✓ All visualizations saved to: /content/downloads/

Regressor Files:
  • reg_01_predicted_vs_actual.png
  • reg_02_residual_analysis.png
  • reg_03_error_by_los_bins.png
  • reg_04_feature_importance.png
  • reg_05_prediction_distributions.png
  • reg_06_model_summary.png

Classifier Files:
  • clf_01_confusion_matrix.png
  • clf_02_roc_curve.png
  • clf_03_precision_recall.png
  • clf_04_feature_importance.png
  • clf_05_probability_distribution.png
  • clf_06_model_summary.png

